# Lunar Eclipse Data Transformation

This notebook creates the derived columns required for the dashboard filters:
`Year` and `Eclipse Category`.


## Load

In [15]:
from pathlib import Path

import pandas as pd

lunar_df = pd.read_csv("../backend/data/raw/lunar.csv")

lunar_df.head()

,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Quincena Solar Eclipse,Gamma,Penumbral Magnitude,Umbral Magnitude,Latitude,Longitude,Penumbral Eclipse Duration (m),Partial Eclipse Duration (m),Total Eclipse Duration (m)
0,1,-1999 June 26,14:13:28,46437,-49456,17,N,t-,-1.0981,0.8791,-0.1922,24S,22W,268.8,-,-
1,2,-1999 November 21,20:23:49,46427,-49451,-16,N,-a,-1.1155,0.8143,-0.1921,15N,98W,233.4,-,-
2,3,-1998 May 17,05:47:36,46416,-49445,-11,P,-t,0.8988,1.2105,0.2069,13S,89E,281.7,102.7,-
3,4,-1998 November 11,05:15:58,46404,-49439,-6,P,-a,-0.4644,2.0382,0.9740,12N,113E,343.4,200.8,-
4,5,-1997 May 6,18:57:01,46392,-49433,-1,T+,pp,0.1003,2.6513,1.6963,11S,92W,322.8,213.5,98.2


## Create `Year` column

Extract the year from `Calendar Date` so it can be used as a dashboard filter.

- LLM Help about the `-` and reason to keep it. (I dont know anything about timelines in eclipse.)
- Backend: keep Year = -1999 because it sorts and filters correctly.
- Streamlit presentation layer: display it as 2000 BCE, without dash
- Keep the numeric Year column unchanged with the `-` and apply the user-friendly label in the frontend.
  
```
| Stored value | Dashboard label |
| -----------: | --------------- |
|      `-1999` | `2000 BCE`      |
|          `0` | `1 BCE`         |
|       `2010` | `2010 CE`       |
```


In [16]:
# Extract the year from the Calendar Date column
lunar_df["Year"] = (
    lunar_df["Calendar Date"]
    .str.split()   # Split each date into separate parts
    .str[0]        # Select the first part, which contains the year
    .astype(int)   # Convert the year from text to an integer
)

# Display the original date and extracted year
lunar_df[["Calendar Date", "Year"]].head()

,Calendar Date,Year
0,-1999 June 26,-1999
1,-1999 November 21,-1999
2,-1998 May 17,-1998
3,-1998 November 11,-1998
4,-1997 May 6,-1997


## Create Eclipse Category
Group the detailed NASA eclipse codes into four user-friendly categories.

In [17]:
# Map each eclipse type code to a category name
category_mapping = {
    "N": "Penumbral",
    "P": "Partial",
    "T": "Total",
}


# Create a new column containing new eclipse category
lunar_df["Eclipse Category"] = (
    lunar_df["Eclipse Type"]        # from the OG data
    .str[0]                         # extract 1st letter of the eclipse type
    .map(category_mapping)          # replace letter with full category name from category mapping dict
)

# display both columns
lunar_df[["Eclipse Type", "Eclipse Category"]].head()

,Eclipse Type,Eclipse Category
0,N,Penumbral
1,N,Penumbral
2,P,Partial
3,P,Partial
4,T+,Total


In [18]:
# display each Eclipse Category with total counts each
# total of 11,898 rows

lunar_df["Eclipse Category"].value_counts()

Eclipse Category
Penumbral    4378
Partial      4207
Total        3479
Name: count, dtype: int64

## Test the filtering logic for the dashboard

In [19]:
# Test by choosing a year and eclipse category
# Should return the correct value
selected_year = 2010
selected_category = "Penumbral"


# Filter the DataFrame using the selected year and category
filtered_df = lunar_df[
    (lunar_df["Year"] == selected_year)
    & (lunar_df["Eclipse Category"] == selected_category)
]


# Count the number of matching solar eclipses with len funciton
len(filtered_df)

0

In [20]:
# To see which eclipses occurred in 2010


lunar_df[lunar_df["Year"] == 2010][
    ["Calendar Date", "Eclipse Type", "Eclipse Category"]
]

,Calendar Date,Eclipse Type,Eclipse Category
9672,2010 June 26,P,Partial
9673,2010 December 21,T,Total


# Save transformed dataset

In [21]:
lunar_df.to_csv(
    "../backend/data/transformed/lunar.csv",
    index=False, # prevents Pandas from adding an unnecessary index column to the CSV.
)

## Check available year range

Check the earliest and latest years available for the dashboard filter.

In [22]:
# Check the minimum and maximum years in the lunar dataset
lunar_df["Year"].agg(["min", "max"])

min   -1999
max    3000
Name: Year, dtype: int64

In [23]:
available_lunar_years = sorted(lunar_df["Year"].unique())

print("Earliest year:", available_lunar_years[0])
print("Latest year:", available_lunar_years[-1])
print("Number of available years:", len(available_lunar_years))

Earliest year: -1999
Latest year: 3000
Number of available years: 5000
